<a href="https://colab.research.google.com/github/simon-mellergaard/GAI-with-LLMs/blob/main/Project%20codes/Assignment05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5

> With point of departure in Chapter 6 in Hands-on Generative AI with Transformers and Diffusion Models, carefully fine-tune mistralai/Mistral-7B-v0.3 while manipulating key hyperparameters. For your best model, present the training-validation plot and completions for the prompts below:

``` python
prompts = [
    """What is the capital of Germany? Explain why thats the case and if it \
    was different in the past?""",
    "Write a Python function to calculate the factorial of a number.",
    """A rectangular garden has a length of 25 feet and a width of 15 feet. \
    If you want to build a fence around the entire garden, how many feet of \
    fencing will you need?""",
    """What is the difference between a fruit and a vegetable? Give examples \
    of each.""",
]
```

## Setup

In [ ]:
# Libraries

# Functions
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from transformers import load_dataset


In [ ]:
# Logging in to Hugging Face and wandb
os.environ['HF_TOKEN'] = userdata.get('HF')
os.environ['WANDB_TOKEN'] = userdata.get('wandb')
os.environ['HF_USER'] = userdata.get('HF_USER')
login_hf(os.environ['HF_TOKEN'])
login_wandb(key = os.environ['WANDB_TOKEN'])

In [ ]:
# Setting up the device (GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

## Section 1

The model used is the [Mistral-7B model](https://huggingface.co/mistralai/Mistral-7B-v0.3), which has more than 7 billion parameters.

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.3",
    quantization_config=quantization_config,
    device_map="auto",
)

# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.3")

In [ ]:
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

In [ ]:
sft_config = SFTConfig(
    "fine_tune_e2e",
    push_to_hub=True,
    per_device_train_batch_size=8, # Might need to reduce
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    num_train_epochs=2,
    eval_strategy="epoch",
    eval_steps=200,
    logging_steps=200,
    gradient_checkpointing=True,
    max_length=512, # Changed from max_seq_length
    # New parameters
    dataset_text_field="text",
    packing=True,
)


trainer = SFTTrainer(
    model,
    args=sft_config,
    train_dataset=dataset.select(range(300)),
    eval_dataset=dataset_eval.select(range(100)),
    peft_config=peft_config,
)

In [ ]:
trainer.train()

In [ ]:
trainer.push_to_hub()